# Notebook Pengujian API Model Serving - Telco Customer Churn
**Nama:** Bayu Frassetyo Wibowo  
**Username Dicoding:** bayufrassetyo  

Notebook ini digunakan untuk menguji endpoint TensorFlow Serving dari model customer_churn_model sesuai dengan kriteria tambahan MLOps Dicoding. Data input dikemas menjadi serialized `tf.train.Example` berekstensi Base64 agar sesuai dengan spesifikasi signature model.


# Memeriksa Koneksi Metadata Server

In [ ]:
import json
import requests

METADATA_URL = "http://localhost:8080/v1/models/customer_churn_model/metadata"

try:
    response = requests.get(METADATA_URL, timeout=10)
    print(f"Status Koneksi: {response.status_code} OK")
    print("\nRespon Metadata dari Server:")
    print(json.dumps(response.json(), indent=2)[:500] + "...\n[Output Truncated]")
except Exception as e:
    print("Server Docker belum aktif di port 8080 lokal. Pastikan container sudah berjalan!")


# Proses Serialisasi Data Churn & Request Prediksi

In [ ]:
import base64
import json
import requests
import tensorflow as tf

SERVING_URL = "http://localhost:8080/v1/models/customer_churn_model:predict"

# 1. Menyiapkan sampel data satu pelanggan baru (Telco Churn Dataset)
sample_customer = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 34.0,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "No",
    "DeviceProtection": "Yes",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "One year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Mailed check",
    "MonthlyCharges": 56.95,
    "TotalCharges": "1889.5"
}

# 2. Fungsi pembantu pembungkus tipe data internal TensorFlow Protobuf
def _int_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))

def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value.encode("utf-8")]))

# 3. Fungsi mengubah dictionary data mentah menjadi Serialized Protocol Buffer
def make_serialized_example(row):
    example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "gender": _bytes_feature(row["gender"]),
                "SeniorCitizen": _int_feature(row["SeniorCitizen"]),
                "Partner": _bytes_feature(row["Partner"]),
                "Dependents": _bytes_feature(row["Dependents"]),
                "tenure": _float_feature(row["tenure"]),
                "PhoneService": _bytes_feature(row["PhoneService"]),
                "MultipleLines": _bytes_feature(row["MultipleLines"]),
                "InternetService": _bytes_feature(row["InternetService"]),
                "OnlineSecurity": _bytes_feature(row["OnlineSecurity"]),
                "OnlineBackup": _bytes_feature(row["OnlineBackup"]),
                "DeviceProtection": _bytes_feature(row["DeviceProtection"]),
                "TechSupport": _bytes_feature(row["TechSupport"]),
                "StreamingTV": _bytes_feature(row["StreamingTV"]),
                "StreamingMovies": _bytes_feature(row["StreamingMovies"]),
                "Contract": _bytes_feature(row["Contract"]),
                "PaperlessBilling": _bytes_feature(row["PaperlessBilling"]),
                "PaymentMethod": _bytes_feature(row["PaymentMethod"]),
                "MonthlyCharges": _float_feature(row["MonthlyCharges"]),
                "TotalCharges": _bytes_feature(row["TotalCharges"])
            }
        )
    )
    return example.SerializeToString()

try:
    # 4. Melakukan kompresi encoding b64 sesuai standar TensorFlow Serving REST API
    serialized_example = make_serialized_example(sample_customer)
    encoded_example = base64.b64encode(serialized_example).decode("utf-8")
    
    payload = {
        "signature_name": "serving_default",
        "instances": [{"examples": {"b64": encoded_example}}],
    }
    
    # 5. Mengirimkan request POST ke server Docker
    response = requests.post(SERVING_URL, json=payload, timeout=30)
    response.raise_for_status()
    result = response.json()
    
    # 6. Menampilkan output probabilitas prediksi dari model
    print("Status Koneksi POST: Sukses")
    print("\nRespon Hasil Prediksi dari Server Docker:")
    print(json.dumps(result, indent=2))
    
    # 7. Ekstraksi keputusan akhir probabilitas klasifikasi biner
    predictions = result.get("predictions", [])
    if predictions:
        prob_vector = predictions[0]
        predicted_idx = max(range(len(prob_vector)), key=lambda idx: prob_vector[idx])
        print("\n=========================================")
        print("Vektor Probabilitas [Tidak Churn, Churn]:", prob_vector)
        print("Hasil Indeks Prediksi Terpilih:", predicted_idx)
        print("Keputusan Model:", "Pelanggan akan CHURN" if predicted_idx == 1 else "Pelanggan Tetap SETIA (Not Churn)")
        print("=========================================")
except Exception as e:
    print("Gagal menembak endpoint prediksi server.")
    print("Pastikan Docker Desktop aktif dan container berjalan dengan port 8080:8501!")
    print(f"Detail Error: {e}")
